In [1]:
print(123)

123


In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 12:05:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.2’

rag_helper.py.2     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-23 12:05:39 (40.7 MB/s) - ‘rag_helper.py.2’ saved [2134/2134]

--2026-06-23 12:05:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 20

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [10]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [11]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

('To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your OS:\n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you get a connection refused error while prompting Ollama RAG, restart the Ollama server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```', 742, 184)


In [12]:
answer

('To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your OS:\n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you get a connection refused error while prompting Ollama RAG, restart the Ollama server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```',
 742,
 184)

In [13]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

('The FAQ doesn’t mention **Olama** specifically, but it does say you can **run the course locally** instead of using Codespaces.\n\nIf you want to run it locally, you should be comfortable setting up:\n\n- Python\n- `uv`\n- Jupyter\n- Docker\n- any other tools needed for the module\n\nAlso, if you run locally, make sure you:\n\n- document your setup\n- keep your environment reproducible\n\nIf you meant a specific local setup for **Ollama**, I don’t have that information in the provided context.', 885, 118)


In [14]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'I can help, but I need a bit more context. Which course are you referring to?\n\nIf you tell me the course name or share a link/screenshot, I can help you figure out:\n- whether enrollment is still open,\n- how to join,\n- any prerequisites or deadlines,\n- and what to do if it’s already started.'

In [15]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [16]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [41]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    
)

In [46]:
print(response)

Response(id='resp_0f6efb1378e35da0006a3a78b427d0819d8efbed8e9f38bba6', created_at=1782216884.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0f6efb1378e35da0006a3a78b66bb4819da11f44423b6da772', content=[ResponseOutputText(annotations=[], text='Maybe — but I’d need a bit more context to answer accurately.\n\nIf you mean a specific course, your ability to join usually depends on:\n- whether enrollment is still open,\n- whether there are prerequisites,\n- if there’s space remaining,\n- and whether the instructor/admin allows late registration.\n\nIf you want, send me the course name or a link, and I can help you figure out the likely next step or draft a message to ask for late enrollment.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], to

In [43]:
print(response.output_text)

Maybe — but I’d need a bit more context to answer accurately.

If you mean a specific course, your ability to join usually depends on:
- whether enrollment is still open,
- whether there are prerequisites,
- if there’s space remaining,
- and whether the instructor/admin allows late registration.

If you want, send me the course name or a link, and I can help you figure out the likely next step or draft a message to ask for late enrollment.


In [45]:
response.output

[ResponseOutputMessage(id='msg_0f6efb1378e35da0006a3a78b66bb4819da11f44423b6da772', content=[ResponseOutputText(annotations=[], text='Maybe — but I’d need a bit more context to answer accurately.\n\nIf you mean a specific course, your ability to join usually depends on:\n- whether enrollment is still open,\n- whether there are prerequisites,\n- if there’s space remaining,\n- and whether the instructor/admin allows late registration.\n\nIf you want, send me the course name or a link, and I can help you figure out the likely next step or draft a message to ask for late enrollment.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [29]:
len(response.output)

1

In [47]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [48]:
print(response.output_text)

In [49]:
len(response.output)

1

In [50]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? enrollment late join discovered the course"}', call_id='call_TdRUYGsId2zwUWvLJwGpEnJk', name='search', type='function_call', id='fc_03ce47b93140907a006a3a79e14234819fbd2e05954486d7b6', namespace=None, status='completed')]

In [51]:
# to remove square brackets
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? enrollment late join discovered the course"}', call_id='call_TdRUYGsId2zwUWvLJwGpEnJk', name='search', type='function_call', id='fc_03ce47b93140907a006a3a79e14234819fbd2e05954486d7b6', namespace=None, status='completed')

In [52]:
call = response.output[0]

In [53]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? enrollment late join discovered the course"}', call_id='call_TdRUYGsId2zwUWvLJwGpEnJk', name='search', type='function_call', id='fc_03ce47b93140907a006a3a79e14234819fbd2e05954486d7b6', namespace=None, status='completed')

In [54]:
call.arguments

'{"query":"Can I join the course after it has started? enrollment late join discovered the course"}'

In [55]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course after it has started? enrollment late join discovered the course'}

In [16]:
call.name

'search'

In [17]:
results = search(**args)

In [18]:
result_json = json.dumps(results, indent=2)

In [19]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [20]:
messages.append(call)

In [21]:
messages.append(function_call_output)

In [22]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course late enrollment"}', call_id='call_ZbjncavtV07o4ti4yacfD8IA', name='search', type='function_call', id='fc_01babd39c78723a8006a394cb4b37c819fab58006bc34661c3', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_ZbjncavtV07o4ti4yacfD8IA',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workfl

In [23]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [24]:
print(response.output_text)

Yes — you can join even if you just discovered it.

You can start learning and submit homework while the submission forms are open. If you want a certificate, though, you need to submit your project while the course is still accepting submissions.

If you want, I can also point you to the best place to start.


In [25]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(776, 68)

In [26]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [27]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [29]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [30]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late join FAQ"}


In [31]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_982uBDuaZkS32bpqemZuRpLK', name='search', type='function_call', id='fc_0a85bd4c1ad20639006a394d3fa32481a08e4363e18057de3c', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_982uBDuaZkS32bpqemZuRpLK',
  'output': '[\n  {\n    "id": "74eb249bb

In [32]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break


iteration #1...
function_call: search {"query":"join the course enrollment can I join course discovered just now"}
function_call: search {"query":"course enrollment late join discovered course can I still join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

You can start learning right away, and if you want a certificate, make sure to submit your project while submissions are still open. Homework is not mandatory, but it’s recommended.

If you want, I can also help you figure out how to get started with the course materials. Are there other areas you want to explore?


In [33]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [35]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course late discovered can I join course enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important caveat: if you want a certificate, you need to submit your project while submissions are still open. If you’re just looking to learn, you can start anytime.

If you want, I can also help you find the course materials, schedule, or how to get started.


In [36]:
result

'Yes — you can still join the course.\n\nOne important caveat: if you want a certificate, you need to submit your project while submissions are still open. If you’re just looking to learn, you can start anytime.\n\nIf you want, I can also help you find the course materials, schedule, or how to get started.'

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry related to “queen gambit,” so it looks like this is off-topic for the course.

If you meant something course-related, feel free to rephrase it, and I can look it up. Are there other areas you want to explore?


In [39]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [40]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [41]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [42]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [43]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [44]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [45]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [46]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [47]:
result.cost

CostInfo(input_cost=Decimal('0.00269325'), output_cost=Decimal('0.0012465'), total_cost=Decimal('0.00393975'))

In [48]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local setup Ollama course FAQ"}', call_

In [49]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run();